# 从NS代理开始(ADK执行).

这本笔记本在单 FHIR 捆绑上通过运行神经系统代理： 构建时间顺序的超长图，预计算节点嵌入，即时处理代理，并在记录上提问。

## 环境搭建

从 仓库 根，安装软件包并设置 GenAI API 密钥：

```bash
pip install -e .
export GENAI_API_KEY="your_gemini_api_key"
```

In [ ]:
import os
import json
import time
import datetime

from ns_agent_adk.core import graph as graph_module
from ns_agent_adk.core import linearizer as linearizer_module
from ns_agent_adk.config import config as config_module
from ns_agent_adk import engine as engine_module

In [ ]:
with open("sample_patient.json") as f:
    fhir_bundle = json.load(f)

print(f"Loaded bundle with {len(fhir_bundle['entry'])} resources.")

## 配置模型

公众GenAIAPI默认使用;设置`GENAI_API_KEY`在环境中。

In [ ]:
API_KEY = os.getenv("GENAI_API_KEY")
if not API_KEY:
    raise ValueError("Set GENAI_API_KEY in your environment.")

config = config_module.Config(
    embedding_api_key=API_KEY,
    llm_api_key=API_KEY,
    embedding_model_name="gemini-embedding-001",
    llm_model_name="gemini-3-flash-preview",
    temporal_parser_model_name="gemini-3-flash-preview",
)

## 步骤 1 : 构建高频图

In [ ]:
graph = graph_module.ChronologicalHypergraph()
graph.build_from_bundle(fhir_bundle)
print(f"Graph built with {len(graph.spine)} hypernodes in the spine.")

In [ ]:
# 图. spine [0] 持有第一个超节点;图. spine [0]. nodes [0] 其第一个节点。

## 步骤 2:预计算节点嵌入

这每张图都运行一次。

In [ ]:
embedder = config.get_embedder()
precomputed_node_embeddings = linearizer_module.precompute_node_embeddings(graph, embedder)
print(f"Computed {len(precomputed_node_embeddings)} node embeddings.")

## 步骤 3: 说明代理并提问

In [ ]:
ns_agent = engine_module.NeuroSymbolicAgent(
    config=config,
    graph=graph,
    precomputed_node_embeddings=precomputed_node_embeddings,
)

In [ ]:
query = "Does my latest cholesterol medication appear to be working?"
final_response, reasoning_trace = ns_agent.execute(query)
print(final_response)

In [ ]:
query = "Who prescribed my cholesterol medication and when?"
final_response, reasoning_trace = ns_agent.execute(query)
print(final_response)

In [ ]:
query = "What is the trend of my blood pressure?"
final_response, reasoning_trace = ns_agent.execute(query)
print(final_response)

## 检查推理线索

In [ ]:
for event in reasoning_trace:
    print(f"\n--- Turn (Author: {event.author}) ---")
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"Text Content:\n{part.text}")
    if event.actions:
        for action in event.actions:
            print(f"Triggered ADK Action / Tool Call: {action}")